# Unit 2: Cryptography - Compiled Period-Finding Toy for N = 15

**Companion notebook for *What Quantum Computers Are Actually For***

**Notebook class:** toy demonstration


**Code note:** This notebook is written as teaching code rather than library code. The Quokka calls, circuit construction, and measurement post-processing stay explicit on purpose so you can inspect the mechanism end to end. A production library would factor more of this behind helpers; here we keep the moving parts visible.

This notebook keeps the classical reduction in Shor's algorithm exact and replaces the large modular-exponentiation circuit with a compiled phase oracle for one eigenphase branch of the $r = 4$ problem.

It does **not** implement general modular exponentiation, and it does **not** claim a faithful end-to-end Shor circuit for arbitrary $N$.

**What you'll see:**
1. The classical reduction from factoring to period-finding
2. A compiled period-finding toy for the $N = 15$, $a = 7$ example
3. Continued fractions recovering the period from the measured phase
4. Classical post-processing turning that period into the factors of 15

In [ ]:
import json
import math
from fractions import Fraction

import numpy as np
import matplotlib.pyplot as plt
import requests
from requests.packages.urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

# --- Quokka connection ---
QUOKKA = "quokka1"
QUOKKA_URL = f"http://{QUOKKA}.quokkacomputing.com/qsim/qasm"

def _decode_quokka_counts(payload: dict) -> dict:
    """Normalize Quokka responses to a simple {bitstring: count} mapping."""
    if isinstance(payload, dict) and all(isinstance(v, int) for v in payload.values()):
        return payload

    if not isinstance(payload, dict):
        raise TypeError(f"Unexpected Quokka payload type: {type(payload)!r}")

    if payload.get("error_code", 0) not in (0, None):
        raise RuntimeError(f"Quokka error {payload.get('error_code')}: {payload.get('error')}")

    result = payload.get("result")
    if not isinstance(result, dict) or "c" not in result:
        raise ValueError(f"Unexpected Quokka result format: {payload}")

    counts = {}
    for shot in result["c"]:
        bitstring = ''.join(str(int(bit)) for bit in shot)
        counts[bitstring] = counts.get(bitstring, 0) + 1
    return counts

def run_qasm(program: str, shots: int = 1024) -> dict:
    result = requests.post(QUOKKA_URL, json={"script": program, "count": shots}, verify=False)
    result.raise_for_status()
    return _decode_quokka_counts(result.json())

print(f"Connected to {QUOKKA_URL}")

## 1. The classical part: factoring → period-finding

We want to factor $N = 15$. Pick $a = 7$ (coprime to 15).

Compute $f(x) = 7^x \bmod 15$ and look for the period:

In [ ]:
N = 15
a = 7

print(f"Factoring N = {N} using a = {a}")
print(f"\nf(x) = {a}^x mod {N}:")
print("-" * 20)

values = []
for x in range(16):
    fx = pow(a, x, N)
    values.append(fx)
    print(f"  f({x:2d}) = {fx}")

# Find the period classically
r_classical = next(x for x in range(1, N) if pow(a, x, N) == 1)
print(f"\nPeriod r = {r_classical}")

## 2. The quantum part: a compiled period-finding toy

For $N = 15$ and $a = 7$, the true order is $r = 4$.

Instead of implementing full modular exponentiation, we compile one eigenphase branch with phase $j/r = 1/4$ into a small QPE-style circuit. That is enough to demonstrate the QFT plus continued-fractions logic honestly, without pretending this is a scalable Shor implementation.

In [ ]:
# Compiled period-finding toy for the j/r = 1/4 branch of the N=15, a=7 example
ancilla_bits = 3
Q = 2 ** ancilla_bits
target_j = 1
phase_exact = target_j / r_classical
phase_integer = int(np.clip(np.rint(phase_exact * Q), 0, Q - 1))
encoded_phase = phase_integer / Q
expected_peak = format(phase_integer, f"0{ancilla_bits}b")
phase_angle = 2 * np.pi * encoded_phase

print(f"Classical period:               r = {r_classical}")
print(f"Chosen eigenphase branch:       j/r = {target_j}/{r_classical}")
print(f"Encoded 3-bit phase fraction:   {encoded_phase:.4f} ({expected_peak})")

qpe_circuit = f"""OPENQASM 2.0;
include "qelib1.inc";

qreg q[4];  // q[0-2] = ancilla register, q[3] = eigenstate register
creg c[3];

// Prepare the one-qubit eigenstate |1>
x q[3];

// Put the ancilla register into superposition
h q[0];
h q[1];
h q[2];

// Compiled controlled-U^(2^k) phases for the j/r = 1/4 branch
cu1({phase_angle:.6f}) q[2], q[3];
cu1({2 * phase_angle:.6f}) q[1], q[3];
cu1({4 * phase_angle:.6f}) q[0], q[3];

// Inverse QFT on the ancilla register
h q[0];
cu1({-np.pi / 2:.6f}) q[0], q[1];
h q[1];
cu1({-np.pi / 4:.6f}) q[0], q[2];
cu1({-np.pi / 2:.6f}) q[1], q[2];
h q[2];

// Bit reversal swap
cx q[0], q[2];
cx q[2], q[0];
cx q[0], q[2];

measure q[0] -> c[0];
measure q[1] -> c[1];
measure q[2] -> c[2];
"""

In [ ]:
results = run_qasm(qpe_circuit, shots=1024)

print('Measurement results:')
for outcome in sorted(results.keys(), key=lambda bitstring: results[bitstring], reverse=True):
    k = int(outcome, 2)
    phase_fraction = k / Q
    print(f"  {outcome}: {results[outcome]:>4} counts -> k = {k} -> phase {phase_fraction:.3f}")

best_outcome = max(results, key=results.get)
peak_probability = results[best_outcome] / sum(results.values())
best_k = int(best_outcome, 2)
phase_estimate = best_k / Q

print()
print(f"Dominant outcome:              {best_outcome} (expected {expected_peak})")
print(f"Recovered phase fraction:      {phase_estimate:.4f}")
print(f"Peak probability:              {peak_probability:.3f}")

## 3. Recover the period with continued fractions

In [ ]:
print(f"Using Q = {Q} and dominant k = {best_k}")

if best_k == 0:
    print('k = 0 gives no information; falling back to the known classical period for this toy.')
    r = r_classical
else:
    frac = Fraction(best_k, Q).limit_denominator(N)
    r_candidate = frac.denominator
    valid_period = pow(a, r_candidate, N) == 1
    print(f"k/Q = {best_k}/{Q} = {frac}")
    print(f"Candidate period from continued fractions: r = {r_candidate}")
    print(f"Check a^r mod N == 1: {'yes' if valid_period else 'no'}")
    r = r_candidate if valid_period and r_candidate > 1 else r_classical

print(f"Recovered period: r = {r}")

## 4. Factor N using the recovered period

In [ ]:
print(f"N = {N}, a = {a}, r = {r}")
print()

if r % 2 != 0:
    print(f"r = {r} is odd, so this branch would fail and we would retry with a different a.")
else:
    x = pow(a, r // 2, N)
    print(f"a^(r/2) = {a}^{r // 2} = {x} (mod {N})")

    if x == N - 1:
        print('a^(r/2) is congruent to -1 mod N, so this branch would fail and we would retry.')
    else:
        factor1 = math.gcd(x - 1, N)
        factor2 = math.gcd(x + 1, N)
        print(f"gcd({x} - 1, {N}) = {factor1}")
        print(f"gcd({x} + 1, {N}) = {factor2}")
        print()
        print(f"15 = {factor1} x {factor2}")

In [ ]:
# Visualise the measurement distribution for the compiled period-finding toy
labels = sorted(results.keys())
counts = [results[label] for label in labels]
k_values = [int(label, 2) for label in labels]
colors = ['#009688' if label == best_outcome else '#B0BEC5' for label in labels]

plt.figure(figsize=(9, 4))
plt.bar(labels, counts, color=colors)
plt.xlabel('Measurement outcome')
plt.ylabel('Counts')
plt.title(f'Compiled period-finding toy for {a}^x mod {N}')
plt.legend(handles=[
    plt.Rectangle((0, 0), 1, 1, color='#009688', label=f'Expected peak at k = {best_k}'),
    plt.Rectangle((0, 0), 1, 1, color='#B0BEC5', label='Other outcomes'),
])
plt.tight_layout()
plt.show()

## What's next?

- [Deep-Dive 2 - Inside Shor's Algorithm](https://github.com/johnazariah/quantum-bottleneck/blob/main/manuscript/04-inside-shors.md): the full QFT and oracle story behind period-finding
- Replace the compiled phase oracle with genuine modular exponentiation to move from this toy to a faithful Shor implementation
- [Unit 3 - Drug Discovery](https://github.com/johnazariah/quantum-bottleneck/blob/main/manuscript/05-drug-discovery.md): where the same phase-estimation machinery reappears in chemistry